In [ ]:
COLLECTIONS = {
    "NO2": {
        "asset": "COPERNICUS/S5P/OFFL/L3_NO2",
        "band": "tropospheric_NO2_column_number_density",
    },
    "CO": {
        "asset": "COPERNICUS/S5P/OFFL/L3_CO",
        "band": "CO_column_number_density",
    },
    "TEMP": {
        "asset": "ECMWF/ERA5/HOURLY",
        "band": "temperature_2m",
    },
    "WIND_U": {
        "asset": "ECMWF/ERA5/HOURLY",
        "band": "u_component_of_wind_10m",
    },
    "WIND_V": {
        "asset": "ECMWF/ERA5/HOURLY",
        "band": "v_component_of_wind_10m",
    },
}

LAT = 28.695
LON = 77.65
START_DATE = '2019-01-01'
END_DATE = '2019-01-31'
OUTPUT_FILE = 'demo_data.csv'
CUSTOM_SCALE = 4000
STEP = 0.5

In [ ]:
import ee
from google.colab import auth

# 1. Authenticate and Initialize
auth.authenticate_user()
ee.Initialize(project='climateconsciousimli')

print('Earth Engine initialized successfully.')

Earth Engine initialized successfully.


In [ ]:
# Flag to control S5P quality masking. Set to False to temporarily disable masking for debugging.
APPLY_S5P_MASK = False

In [ ]:
# def export_data_with_offset(name, config):
#     # Define the geometry based on current LAT, LON, and STEP
#     region = ee.Geometry.Rectangle([
#         LON - STEP,
#         LAT - STEP,
#         LON + STEP,
#         LAT + STEP
#     ])

#     # Load collection and filter by date
#     collection = ee.ImageCollection(config['asset']) \
#         .filterDate(START_DATE, END_DATE) \
#         .select(config['band'])

#     # Get native scale and add 100m offset
#     first_img = ee.Image(collection.first())
#     native_scale = first_img.projection().nominalScale().getInfo()
#     process_scale = native_scale + 100

#     # Apply S5P specific quality masking using server-side logic
#     if 'S5P' in config['asset']:
#         def mask_s5p(img):
#             has_qa = img.bandNames().contains('qa_value')
#             qa = img.select('qa_value')
#             return ee.Image(ee.Algorithms.If(has_qa, img.updateMask(qa.gte(0.5)), img))
#         collection = collection.map(mask_s5p)

#     def reduce_image(image):
#         # Add latitude and longitude bands to the image itself
#         img_with_coords = image.addBands(ee.Image.pixelLonLat())

#         # Reduce region to get all properties including lat/lon and the band data
#         mean_props = img_with_coords.reduceRegion(
#             reducer=ee.Reducer.mean(),
#             geometry=region,
#             scale=process_scale,
#             bestEffort=True
#         )

#         # Extract values
#         band_value = mean_props.get(config['band'])
#         mean_lat = mean_props.get('latitude')
#         mean_lon = mean_props.get('longitude')

#         # Create a new dictionary with the desired property names
#         output_properties = ee.Dictionary({
#             'timestamp': image.date().format('YYYY-MM-dd HH:mm'),
#             name: band_value,  # Use the friendly name from COLLECTIONS key
#             'latitude': mean_lat,
#             'longitude': mean_lon
#         })

#         return ee.Feature(None, output_properties)

#     # Map the reduction over the collection
#     features = collection.map(reduce_image)

#     # Export to Google Drive
#     task = ee.batch.Export.table.toDrive(
#         collection=features,
#         description=f'Export_{name}_True_Coords',
#         fileFormat='CSV',
#         fileNamePrefix=f'{name}_data_with_pixel_coords_{int(process_scale)}m'
#     )
#     task.start()
#     print(f'Started task for {name}. Scale: {process_scale:.1f}m. Including pixel-level coordinates.')

# # Execute all exports
# for name, config in COLLECTIONS.items():
#     try:
#         export_data_with_offset(name, config)
#     except Exception as e:
#         print(f'Error starting {name}: {e}')

In [ ]:
def export_data_with_offset(name, config):
    # Define the geometry based on current LAT, LON, and STEP
    region = ee.Geometry.Rectangle([
        LON - STEP,
        LAT - STEP,
        LON + STEP,
        LAT + STEP
    ])

    # Load collection and filter by date
    collection = ee.ImageCollection(config['asset']) \
        .filterDate(START_DATE, END_DATE) \
        .select(config['band'])

    # Get native scale and add 100m offset
    first_img = ee.Image(collection.first())
    native_scale = first_img.projection().nominalScale().getInfo()
    process_scale = native_scale + 100

    # Apply S5P specific quality masking using server-side logic, if the flag is enabled
    if 'S5P' in config['asset'] and APPLY_S5P_MASK:
        def mask_s5p(img):
            has_qa = img.bandNames().contains('qa_value')
            qa = img.select('qa_value')
            return ee.Image(ee.Algorithms.If(has_qa, img.updateMask(qa.gte(0.5)), img))
        collection = collection.map(mask_s5p)

    def sample_and_format_pixels(image):
        # Sample pixels within the region
        sampled_features = image.sampleRegions(
            collection=region,
            scale=process_scale,
            geometries=True  # Include point geometries for each sample
        )

        # Get the timestamp for this image
        timestamp = image.date().format('YYYY-MM-dd HH:mm')

        # Map over the sampled features to add timestamp and rename the band property
        def format_feature(feature):
            # Extract lat/lon from the geometry
            lon = feature.geometry().coordinates().get(0)
            lat = feature.geometry().coordinates().get(1)

            # Create a new dictionary with the desired property names
            properties = ee.Dictionary({
                'timestamp': timestamp,
                name: feature.get(config['band']), # Rename the band
                'latitude': lat,
                'longitude': lon
            })
            return ee.Feature(None, properties) # Create a new feature with desired properties

        return sampled_features.map(format_feature)

    # Map the sampling and formatting function over the image collection and flatten the results
    all_formatted_features = collection.map(sample_and_format_pixels).flatten()

    # Export to Google Drive
    task = ee.batch.Export.table.toDrive(
        collection=all_formatted_features,
        description=f'Export_{name}_Pixel_Coords',
        fileFormat='CSV',
        fileNamePrefix=f'{name}_manual_{int(process_scale)}m',
        selectors=['timestamp', name, 'latitude', 'longitude'] # Explicitly select and order columns
    )
    task.start()
    print(f'Started task for {name}. Scale: {process_scale:.1f}m. Including pixel-level coordinates.')

# Execute all exports
for name, config in COLLECTIONS.items():
    try:
        export_data_with_offset(name, config)
    except Exception as e:
        print(f'Error starting {name}: {e}')

Started task for NO2. Scale: 1213.2m. Including pixel-level coordinates.
Started task for CO. Scale: 1213.2m. Including pixel-level coordinates.
Started task for TEMP. Scale: 27929.9m. Including pixel-level coordinates.
Started task for WIND_U. Scale: 27929.9m. Including pixel-level coordinates.
Started task for WIND_V. Scale: 27929.9m. Including pixel-level coordinates.


In [ ]:
# 3. Monitor Task Status
import pandas as pd
import ee
tasks = ee.batch.Task.list()
status_list = []
for t in tasks[:5]:
    status_list.append({
        'id': t.id,
        'description': t.config.get('description'),
        'state': t.state
    })

display(pd.DataFrame(status_list))

,id,description,state
0,IA3TFGSMJZUONIF2BCTOVAY2,Export_WIND_V_Pixel_Coords,READY
1,QNVDZBNVN7WKBXZNUVK3XV56,Export_WIND_U_Pixel_Coords,READY
2,DQOF4PFWN4YZ7W6NMBUTAILW,Export_TEMP_Pixel_Coords,COMPLETED
3,QPXRM6SFRPTAAU4B66YJJ7YM,Export_CO_Pixel_Coords,COMPLETED
4,V6HBNYZMUQYHREG5U37CGLNV,Export_NO2_Pixel_Coords,COMPLETED


In [ ]:
import time
from datetime import datetime

def monitor_detailed_tasks(limit=10):
    tasks = ee.batch.Task.list()
    detailed_status = []

    for t in tasks[:limit]:
        # Convert milliseconds timestamps to readable dates if they exist
        start_time = t.status().get('start_timestamp_ms')
        update_time = t.status().get('update_timestamp_ms')

        start_dt = datetime.fromtimestamp(start_time/1000).strftime('%Y-%m-%d %H:%M:%S') if start_time else 'N/A'
        update_dt = datetime.fromtimestamp(update_time/1000).strftime('%Y-%m-%d %H:%M:%S') if update_time else 'N/A'

        detailed_status.append({
            'Description': t.config.get('description'),
            'State': t.state,
            'Created': start_dt,
            'Last Update': update_dt,
            'ID': t.id
        })

    df_tasks = pd.DataFrame(detailed_status)
    display(df_tasks)

# Run the detailed monitor
monitor_detailed_tasks()

,Description,State,Created,Last Update,ID
0,Export_WIND_V_Pixel_Coords,READY,N/A,2026-04-22 10:58:23,IA3TFGSMJZUONIF2BCTOVAY2
1,Export_WIND_U_Pixel_Coords,READY,N/A,2026-04-22 10:58:23,QNVDZBNVN7WKBXZNUVK3XV56
2,Export_TEMP_Pixel_Coords,READY,N/A,2026-04-22 10:58:19,DQOF4PFWN4YZ7W6NMBUTAILW
3,Export_CO_Pixel_Coords,READY,N/A,2026-04-22 10:58:19,QPXRM6SFRPTAAU4B66YJJ7YM
4,Export_NO2_Pixel_Coords,READY,N/A,2026-04-22 10:58:19,V6HBNYZMUQYHREG5U37CGLNV
5,Export_WIND_V_Pixel_Coords,CANCELLED,N/A,2026-04-22 10:57:30,AVKOASN37OXMV5ZWDFATWXTL
6,Export_WIND_U_Pixel_Coords,CANCELLED,N/A,2026-04-22 10:57:30,JIK6YFNIKAOG56S3T6FG4NRJ
7,Export_TEMP_Pixel_Coords,CANCELLED,N/A,2026-04-22 10:57:30,3WG7CYN5LH53IP6WIEE6NWXC
8,Export_CO_Pixel_Coords,CANCELLED,N/A,2026-04-22 10:57:31,IIOX2A6GSBM2NM6AEZPFZIPC
9,Export_NO2_Pixel_Coords,CANCEL_REQUESTED,2026-04-22 10:54:57,2026-04-22 10:57:38,GTXVB5VWEKOKWWHDT276ASNC


In [ ]:
# import ee

# # List all tasks
# tasks = ee.batch.Task.list()

# # Filter and cancel those that are queued or currently running
# cancelled_count = 0
# for task in tasks:
#     if task.state in ['READY', 'RUNNING']:
#         task.cancel()
#         print(f'Cancelled: {task.config["description"]} ({task.id})')
#         cancelled_count += 1

# if cancelled_count == 0:
#     print('No active tasks found to cancel.')
# else:
#     print(f'\nSuccessfully requested cancellation for {cancelled_count} tasks.')